In [25]:
import os
import yaml
import pandas as pd
import numpy as np

with open("../../config.local.yaml", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]

ANALYSIS_FILENAME = "tax_analysis_panel.parquet"


In [26]:
regs_df = pd.read_csv(os.path.join(DATA_PATH, "best_treatment_dates.csv"))
tax_df = pd.read_excel(os.path.join(DATA_PATH, "FiSC-Full-Dataset-2023-Update.xlsx"), sheet_name="Data")

In [27]:
# clean dates
regs_df['best_enforcement'] = pd.to_datetime(regs_df['best_enforcement'], errors='coerce')
regs_df['best_passage'] = pd.to_datetime(regs_df['best_passage'], errors='coerce')# clean city names

In [28]:
# Clean city names
mask = tax_df['city_name'] == "TX: Ft. Worth"
tax_df.loc[mask, 'city_name'] = "TX: Fort Worth"

mask = tax_df['city_name'] == "OK: Oklahoma"
tax_df.loc[mask, 'city_name'] = "OK: Oklahoma City"

regs_df['city_name'] = regs_df['state'] + ": " + regs_df['city']

In [29]:
# check match quality

regs_cities = set(regs_df['city_name'].unique())
tax_cities = set(tax_df['city_name'].unique())

assert regs_cities.issubset(tax_cities)


In [ ]:
# make tax data onto regulatory data

df = regs_df.merge(tax_df, on='city_name', how='left')


In [ ]:
# time measures

df['enforcement_year'] = df['best_enforcement'].dt.year
df['passage_year'] = df['best_passage'].dt.year
df['years_from_enforcement'] = (df['year'] - df['enforcement_year'])
df['years_from_passage'] = (df['year'] - df['passage_year'])


/var/folders/ln/_cyxjf216xsc39dhb2hsjdz00000gn/T/ipykernel_46800/2388794089.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['enforcement_year'] = df['best_enforcement'].dt.year
/var/folders/ln/_cyxjf216xsc39dhb2hsjdz00000gn/T/ipykernel_46800/2388794089.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['passage_year'] = df['best_passage'].dt.year
/var/folders/ln/_cyxjf216xsc39dhb2hsjdz00000gn/T/ipykernel_46800/2388794089.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

In [32]:
# for cities with an enforcement date, drop rows >12 years to/from enforcement
mask = (np.abs(df['years_from_enforcement']) > 12) & (df['years_from_enforcement'].notna())
df = df.loc[~mask].reset_index(drop=True)

# for cities without an enforcement date, drop years outside the min/max of the remaining data
year_min = df.loc[df['best_enforcement'].notna(), 'year'].min()
year_max = df.loc[df['best_enforcement'].notna(), 'year'].max()
mask = (df['year'] < year_min) | (df['year'] > year_max)
df = df.loc[~mask].reset_index(drop=True)


In [ ]:
# change enforcement and passage year to 0 for cities without enforcement/passage dates
# (standard convention for CSDID package in R)
df.loc[df['best_enforcement'].isna(), 'enforcement_year'] = 0
df.loc[df['best_passage'].isna(), 'passage_year'] = 0
df['enforcement_year'] = df['enforcement_year'].astype(int)
df['passage_year'] = df['passage_year'].astype(int)


In [34]:
# make a city_id integer
df['city_id'] = df['city_name'].astype('category').cat.codes

/var/folders/ln/_cyxjf216xsc39dhb2hsjdz00000gn/T/ipykernel_46800/2434122789.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['city_id'] = df['city_name'].astype('category').cat.codes


In [ ]:
# output dataframe for analysis
df.to_parquet(os.path.join(DATA_PATH, ANALYSIS_FILENAME))
df.info()